# Add informacion a Big Query

In [1]:
import pandas as pd
import os
from datetime import datetime
import pytz
from dotenv import load_dotenv
from google.cloud import bigquery
import pandas_gbq

pd.set_option("display.max_columns", None)

D:\03_PROYECTOS\ESTUDIOS_AUTO\MLOps-Bootcamp\Modulo_6-Implementacion_y_despliegue_con_GCP\SESION_4\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
D:\03_PROYECTOS\ESTUDIOS_AUTO\MLOps-Bootcamp\Modulo_6-Implementacion_y_despliegue_con_GCP\SESION_4\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  w

In [2]:
load_dotenv()

project_id = os.getenv("GCP_PROJECT_ID")
dataset_id = os.getenv("BQ_DATASET_ID_FEATURES")
table_id = os.getenv("BQ_TABLE_ID_FEATURES")
credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

print("Proyecto:", project_id)
print("Dataset:", dataset_id)
print("Tabla:", table_id)
print("Credencial:", credentials_path)

Proyecto: gcp-processing-vertex-prod-us
Dataset: dev_table
Tabla: features_input
Credencial: secrets/SA-gcp-processing-vertex-prod-us-45b7ce3e8f13.json


In [7]:
df = pd.read_csv('./data/features.txt')
df

,features
0,mssubclass
1,mszoning
2,lotfrontage
3,lotarea
4,neighborhood
5,overallqual
6,overallcond
7,yearbuilt
8,yearremodadd
9,bsmtqual


In [8]:
ZPeru = pytz.timezone("America/Lima")
fecha_carga = datetime.now(ZPeru)
df['date_subida_local'] = fecha_carga.replace(tzinfo=None)
df['date_subida_utc'] = fecha_carga.astimezone(pytz.UTC)
df.head()

,features,date_subida_local,date_subida_utc
0,mssubclass,2026-08-05 01:16:06.404851,2026-08-05 06:16:06.404851+00:00
1,mszoning,2026-08-05 01:16:06.404851,2026-08-05 06:16:06.404851+00:00
2,lotfrontage,2026-08-05 01:16:06.404851,2026-08-05 06:16:06.404851+00:00
3,lotarea,2026-08-05 01:16:06.404851,2026-08-05 06:16:06.404851+00:00
4,neighborhood,2026-08-05 01:16:06.404851,2026-08-05 06:16:06.404851+00:00


In [9]:
destination_table = f"{dataset_id}.{table_id}"
full_table_id = f"{project_id}.{destination_table}"
print(destination_table)

client = bigquery.Client(project=project_id)


tabla_truncada = False
cantidad_esperada = len(df)

try:
    query_truncate = f"""
        TRUNCATE TABLE `{full_table_id}`
    """
    print("Ejecutando TRUNCATE TABLE...")

    truncate_job = client.query(query_truncate)
    truncate_job.result()

    tabla_truncada = True
    print("Tabla vaciada correctamente ✅")

    print("Iniciando carga del DataFrame...")

    pandas_gbq.to_gbq(df , destination_table, if_exists='append', project_id=project_id)

    print("Proceso de carga finalizado.")

    # ---------------------------------------

    query_validacion = f"""
        SELECT COUNT(*) AS cantidad_registros
        FROM `{full_table_id}`
    """

    print("Validando cantidad de registros...")

    query_job = client.query(query_validacion)
    resultado = query_job.result()

    cantidad_cargada = next(resultado).cantidad_registros

    print(f"Registros esperados: {cantidad_esperada}")
    print(f"Registros cargados: {cantidad_cargada}")

    if cantidad_cargada == cantidad_esperada:

        print("Carga realizada y validada correctamente ✅")

    else:

        print("La cantidad de registros no coincide ❌")
        print("Eliminando la información cargada...")

        rollback_job = client.query(query_truncate)
        rollback_job.result()

        print("La tabla fue vaciada nuevamente 🧹")

        raise ValueError(
            "Validación fallida. "
            f"Se esperaban {cantidad_esperada} registros, "
            f"pero BigQuery contiene {cantidad_cargada}."
        )


except Exception as error:

    print(f"Se produjo un error durante el proceso: {error}")

    # Si la tabla ya fue truncada, intentamos dejarla vacía.
    if tabla_truncada:

        try:

            print("Ejecutando limpieza de seguridad...")

            rollback_job = client.query(
                f"TRUNCATE TABLE `{full_table_id}`"
            )
            rollback_job.result()

            print("La tabla quedó vacía después del error 🧹")

        except Exception as error_limpieza:

            print(
                "No se pudo realizar la limpieza de seguridad. "
                f"Detalle: {error_limpieza}"
            )

    raise

dev_table.features_input
Ejecutando TRUNCATE TABLE...


Tabla vaciada correctamente ✅
Iniciando carga del DataFrame...
Proceso de carga finalizado.
Validando cantidad de registros...
Registros esperados: 23
Registros cargados: 23
Carga realizada y validada correctamente ✅
